# Experiment: Oracle Beta Search

Objective:
- Use the same six application-style scenarios as the cross-method notebook.
- Initialize beta from a pilot-only rule, without the discrete scan.
- Run Optuna HPO over beta multiplier and swap size to obtain an oracle scenario-specific beta.
- Save the best oracle beta and compare it against the pilot-only initialization.

In [1]:
from __future__ import annotations

from dataclasses import replace
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "perm_pval").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing perm_pval/ and results/.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.environ.setdefault("MPLCONFIGDIR", str(project_root / ".matplotlib"))

from perm_pval.experiments.notebook_studies import (
    BetaSweepStudyConfig,
    CrossMethodStudyConfig,
    DEFAULT_MCMC_OBJECTIVE_GRID_Q_MULTIPLIERS,
    DEFAULT_MCMC_OBJECTIVE_GRID_SWAP_COUNTS,
    MCMCWorkflowConfig,
    MCMC_OBJECTIVE_GRID_REALISTIC_OBJECTIVES,
    SAMCWorkflowConfig,
    _mcmc_eval_count,
    _steps_per_chain,
    build_beta_initialization,
    build_beta_workflow,
    create_timestamped_run_dir,
    load_beta_sweep_saved_output,
    load_cross_method_saved_output,
    load_mcmc_objective_grid_saved_output,
    load_selected_scenarios,
    run_named_mcmc_checkpoint_study,
    run_mcmc_objective_grid_study,
    save_mcmc_objective_grid_outputs,
    regenerate_beta_sweep_plots_from_saved,
    regenerate_cross_method_plots_from_saved,
    run_beta_checkpoint_study,
    run_cross_method_study,
    save_beta_sweep_outputs,
    save_cross_method_outputs,
    summarize_records,
)

pd.set_option("display.max_columns", 100)
project_root

PosixPath('/Users/noamchowers/Documents/University/Thesis/Code/MCMCIS')

In [2]:
import json
import numpy as np
import optuna
from optuna.samplers import TPESampler

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

`ESTIMATION_POINTS` are the final evaluation checkpoints for the pilot-initialized and oracle-selected configurations.  
The Optuna objective itself is evaluated at `HPO_CHECKPOINT`, which is treated as an extra pre-production calibration budget and is not part of the article x-axis budget.

In [3]:
FAST_MODE = False
SAVE_OUTPUTS = True

CATALOG_PATH = project_root / "results" / "exact_scenarios" / "v1" / "catalog.json"
OUTPUT_ROOT = project_root / "results" / "mcmcis_beta_notebook"

SCENARIO_GROUP = "threshold_suite"
SCENARIO_KEYS_OVERRIDE = [
    "gwas_additive_score_ultra_n60",
    "poisson_diffmeans_hep_ultra_n200",
    "gwas_additive_score_sig_n60",
    "poisson_diffmeans_hep_sig_n200",
    "gwas_additive_score_above_n60",
    "poisson_diffmeans_hep_above_n200",
]
ESTIMATION_POINTS = (500_000, 1_000_000, 2_500_000, 5_000_000) if not FAST_MODE else (200_000,)
HPO_CHECKPOINT = max(ESTIMATION_POINTS)
HPO_N_TRIALS = 10 if not FAST_MODE else 5
HPO_TRIAL_REPEATS = 3 if not FAST_MODE else 1
HPO_OBJECTIVE_METRIC = "rmse"
HPO_BETA_MULTIPLIER_LOW = 0.001
HPO_BETA_MULTIPLIER_HIGH = 4.0
HPO_SWAP_CHOICES_DEFAULT = (1, 2, 3)
HPO_SWAP_CHOICES_BY_SETTING = {
    "gwas_threshold_suite": (1, 2),
    "hep_threshold_suite": (1, 2, 4, 6, 8),
}
DEFAULT_BASELINE_SWAP_BY_SETTING = {
    "gwas_threshold_suite": 2,
    "hep_threshold_suite": 6,
}
FINAL_EVAL_REPEATS = 5 if not FAST_MODE else 2
FINAL_EVAL_N_JOBS = min(FINAL_EVAL_REPEATS, os.cpu_count() or 1)
MIN_TAIL_STATES = 2
BASE_SEED = 54_321

init_cfg = MCMCWorkflowConfig(
    use_true_p0_for_q_target=False,
    p0_guess=1e-8,
    pilot_samples=20_000 if not FAST_MODE else 1_000,
    scale_method="sd",
    beta_max_init=1e6,
    chains=2,
    thin=1,
    estimate_variance=False,
    proposal_size=1,
)

hpo_eval_cfg = BetaSweepStudyConfig(
    estimation_points=(HPO_CHECKPOINT,),
    repeats=HPO_TRIAL_REPEATS,
    beta_multipliers=(1.0,),
    chains=3,
    thin=1,
    estimate_variance=False,
    chain_n_jobs=3,
    proposal_size=1,
    base_seed=BASE_SEED,
    n_jobs=1,
)

final_eval_cfg = BetaSweepStudyConfig(
    estimation_points=ESTIMATION_POINTS,
    repeats=FINAL_EVAL_REPEATS,
    beta_multipliers=(1.0,),
    chains=2,
    thin=1,
    estimate_variance=True,
    proposal_size=1,
    base_seed=BASE_SEED + 500_000,
    n_jobs=FINAL_EVAL_N_JOBS,
)

def reference_p0_for_scenario(scenario) -> float:
    threshold = scenario.extra.get("known_significance_threshold")
    if threshold is not None:
        threshold_f = float(threshold)
        if threshold_f == threshold_f and threshold_f > 0.0:
            return threshold_f
    return float(scenario.exact_p)


def swap_choices_for_scenario(scenario) -> tuple[int, ...]:
    setting_key = str(scenario.extra.get("application_setting_key", ""))
    if setting_key in HPO_SWAP_CHOICES_BY_SETTING:
        return tuple(int(v) for v in HPO_SWAP_CHOICES_BY_SETTING[setting_key])
    return tuple(int(v) for v in HPO_SWAP_CHOICES_DEFAULT)


def baseline_swap_for_scenario(scenario) -> int:
    setting_key = str(scenario.extra.get("application_setting_key", ""))
    if setting_key in DEFAULT_BASELINE_SWAP_BY_SETTING:
        return int(DEFAULT_BASELINE_SWAP_BY_SETTING[setting_key])
    return int(swap_choices_for_scenario(scenario)[0])


def trial_objective_value(summary_row: dict, metric: str) -> float:
    metric_val = float(summary_row[metric])
    if not np.isfinite(metric_val):
        return float("inf")
    return metric_val


print(json.dumps({
    "FAST_MODE": FAST_MODE,
    "SCENARIO_GROUP": SCENARIO_GROUP,
    "SCENARIO_KEYS_OVERRIDE": SCENARIO_KEYS_OVERRIDE,
    "ESTIMATION_POINTS": ESTIMATION_POINTS,
    "HPO_CHECKPOINT": HPO_CHECKPOINT,
    "HPO_N_TRIALS": HPO_N_TRIALS,
    "HPO_TRIAL_REPEATS": HPO_TRIAL_REPEATS,
    "HPO_OBJECTIVE_METRIC": HPO_OBJECTIVE_METRIC,
    "HPO_BETA_MULTIPLIER_LOW": HPO_BETA_MULTIPLIER_LOW,
    "HPO_BETA_MULTIPLIER_HIGH": HPO_BETA_MULTIPLIER_HIGH,
    "HPO_SWAP_CHOICES_BY_SETTING": HPO_SWAP_CHOICES_BY_SETTING,
    "HPO_CHAINS_PER_TRIAL": int(hpo_eval_cfg.chains),
    "HPO_CHAIN_N_JOBS": int(hpo_eval_cfg.chain_n_jobs),
    "FINAL_EVAL_REPEATS": FINAL_EVAL_REPEATS,
    "FINAL_EVAL_N_JOBS": FINAL_EVAL_N_JOBS,
    "SAVE_OUTPUTS": SAVE_OUTPUTS,
    "REFERENCE_P0_MODE": "known_significance_threshold_else_exact_p",
}, indent=2))

{
  "FAST_MODE": false,
  "SCENARIO_GROUP": "threshold_suite",
  "SCENARIO_KEYS_OVERRIDE": [
    "gwas_additive_score_ultra_n60",
    "poisson_diffmeans_hep_ultra_n200",
    "gwas_additive_score_sig_n60",
    "poisson_diffmeans_hep_sig_n200",
    "gwas_additive_score_above_n60",
    "poisson_diffmeans_hep_above_n200"
  ],
  "ESTIMATION_POINTS": [
    500000,
    1000000,
    2500000,
    5000000
  ],
  "HPO_CHECKPOINT": 5000000,
  "HPO_N_TRIALS": 10,
  "HPO_TRIAL_REPEATS": 3,
  "HPO_OBJECTIVE_METRIC": "rmse",
  "HPO_BETA_MULTIPLIER_LOW": 0.001,
  "HPO_BETA_MULTIPLIER_HIGH": 4.0,
  "HPO_SWAP_CHOICES_BY_SETTING": {
    "gwas_threshold_suite": [
      1,
      2
    ],
    "hep_threshold_suite": [
      1,
      2,
      4,
      6,
      8
    ]
  },
  "HPO_CHAINS_PER_TRIAL": 3,
  "HPO_CHAIN_N_JOBS": 3,
  "FINAL_EVAL_REPEATS": 5,
  "FINAL_EVAL_N_JOBS": 5,
  "SAVE_OUTPUTS": true,
  "REFERENCE_P0_MODE": "known_significance_threshold_else_exact_p"
}


## Load Scenarios

In [4]:
scenarios = load_selected_scenarios(
    catalog_path=CATALOG_PATH,
    scenario_keys=SCENARIO_KEYS_OVERRIDE,
    portfolio_group=None if SCENARIO_KEYS_OVERRIDE is not None else SCENARIO_GROUP,
    min_tail_states=MIN_TAIL_STATES,
)

run_dir = create_timestamped_run_dir(OUTPUT_ROOT, "beta_diag") if SAVE_OUTPUTS else None

pd.DataFrame([
    {
        "scenario": s.key,
        "exact_p": s.exact_p,
        "tail_hits": s.exact_tail_hits,
        "n_perm": s.exact_n_perm,
        "exact_method": s.exact_method,
        "family": s.portfolio.get("family"),
        "rarity_band": s.portfolio.get("rarity_band"),
        "difficulty": s.portfolio.get("expected_difficulty"),
    }
    for s in scenarios
])

,scenario,exact_p,tail_hits,n_perm,exact_method,family,rarity_band,difficulty
0,gwas_additive_score_ultra_n60,5.096683e-09,602757056,118264581564861424,LinearStatisticDPSolver,gwas_additive_score,ultra_rare,moderate
1,poisson_diffmeans_hep_ultra_n200,3.759363e-09,3404047068642153314749889029354901204235267136...,9054851465610328116540417707748416387450458967...,LinearStatisticDPSolver,poisson_count,ultra_rare,moderate
2,gwas_additive_score_sig_n60,4.561067e-08,5394127080,118264581564861424,LinearStatisticDPSolver,gwas_additive_score,extreme,moderate
3,poisson_diffmeans_hep_sig_n200,2.533610e-07,2294146183289570997753507502932653227347376762...,9054851465610328116540417707748416387450458967...,LinearStatisticDPSolver,poisson_count,extreme,moderate
4,gwas_additive_score_above_n60,6.283615e-07,74312911280,118264581564861424,LinearStatisticDPSolver,gwas_additive_score,extreme,moderate
5,poisson_diffmeans_hep_above_n200,7.668784e-07,6943969698094447475781867186094253216914531992...,9054851465610328116540417707748416387450458967...,LinearStatisticDPSolver,poisson_count,extreme,moderate


## Run Oracle Beta HPO

In [ ]:
oracle_results = {}

for scenario_idx, scenario in enumerate(scenarios):
    scenario_seed = BASE_SEED + 50_000 * scenario_idx
    p0_reference = reference_p0_for_scenario(scenario)
    init_payload = build_beta_initialization(
        scenario.problem,
        scenario.exact_p,
        init_cfg,
        seed=scenario_seed,
        p0_reference=p0_reference,
    )
    beta_init = float(init_payload["beta0_laplace"])
    sigma_t = float(init_payload["sigma_t"])
    swap_choices = swap_choices_for_scenario(scenario)
    baseline_swap = baseline_swap_for_scenario(scenario)
    trial_rows = []
    scenario_dir = (run_dir / scenario.key) if (SAVE_OUTPUTS and run_dir is not None) else None
    if scenario_dir is not None:
        scenario_dir.mkdir(parents=True, exist_ok=True)

    print(json.dumps({
        "scenario": scenario.key,
        "exact_p": scenario.exact_p,
        "application_setting_key": scenario.extra.get("application_setting_key"),
        "known_significance_threshold": scenario.extra.get("known_significance_threshold"),
        "reference_p0": p0_reference,
        "beta0_formula": init_payload["beta0_formula"],
        "beta0_laplace": beta_init,
        "q_target": init_payload["q_target"],
        "sigma_t": sigma_t,
        "swap_choices": swap_choices,
        "baseline_swap": baseline_swap,
    }, indent=2))

    def objective(trial):
        beta_multiplier = float(
            trial.suggest_float(
                "beta_multiplier",
                HPO_BETA_MULTIPLIER_LOW,
                HPO_BETA_MULTIPLIER_HIGH,
                log=True,
            )
        )
        proposal_size = int(trial.suggest_categorical("proposal_size", list(swap_choices)))
        beta = float(beta_init * beta_multiplier)
        trial_eval = run_named_mcmc_checkpoint_study(
            scenario.problem,
            scenario.exact_p,
            config_specs=[
                {
                    "label": "oracle_trial",
                    "config_id": f"trial_{trial.number}",
                    "beta": beta,
                    "proposal_size": proposal_size,
                    "source": "oracle_hpo",
                }
            ],
            sigma_t=sigma_t,
            estimation_points=(HPO_CHECKPOINT,),
            repeats=HPO_TRIAL_REPEATS,
            base_seed=BASE_SEED + 1_000_000 * scenario_idx + 10_000 * trial.number,
            template_cfg=hpo_eval_cfg,
            n_jobs=int(hpo_eval_cfg.n_jobs),
        )
        summary_row = dict(pd.DataFrame(trial_eval["summary"]).iloc[0])
        objective_value = trial_objective_value(summary_row, HPO_OBJECTIVE_METRIC)
        trial_payload = {
            "trial_number": int(trial.number),
            "scenario": scenario.key,
            "beta_multiplier": beta_multiplier,
            "beta": beta,
            "proposal_size": proposal_size,
            "objective_metric": HPO_OBJECTIVE_METRIC,
            "objective_value": float(objective_value),
            "reference_p0": float(p0_reference),
            **summary_row,
        }
        trial_rows.append(trial_payload)
        if scenario_dir is not None:
            pd.DataFrame(trial_rows).to_json(
                scenario_dir / "hpo_trials.jsonl",
                orient="records",
                lines=True,
            )
            (scenario_dir / "hpo_status.json").write_text(
                json.dumps(
                    {
                        "scenario": scenario.key,
                        "completed_trials": len(trial_rows),
                        "target_trials": int(HPO_N_TRIALS),
                        "latest_trial_number": int(trial.number),
                    },
                    indent=2,
                ),
                encoding="utf-8",
            )
        for key, value in trial_payload.items():
            if isinstance(value, (str, int, float)) and not isinstance(value, bool):
                trial.set_user_attr(str(key), value)
        return float(objective_value)

    def persist_hpo_state(study, trial):
        if scenario_dir is None or len(study.trials) == 0:
            return
        best_trial = study.best_trial
        best_payload = {
            "scenario": scenario.key,
            "reference_p0": float(p0_reference),
            "beta_init": float(beta_init),
            "beta0_formula": float(init_payload["beta0_formula"]),
            "beta0_laplace": float(init_payload["beta0_laplace"]),
            "sigma_t": float(sigma_t),
            "best_beta_multiplier": float(best_trial.params["beta_multiplier"]),
            "best_beta": float(beta_init * float(best_trial.params["beta_multiplier"])),
            "best_proposal_size": int(best_trial.params["proposal_size"]),
            "best_objective_value": float(study.best_value),
            "best_trial_number": int(best_trial.number),
            "completed_trials": int(len(study.trials)),
        }
        (scenario_dir / "best_config.partial.json").write_text(
            json.dumps(best_payload, indent=2),
            encoding="utf-8",
        )
        pd.DataFrame(
            [
                {
                    "trial_number": int(t.number),
                    "state": str(t.state),
                    "value": float(t.value) if t.value is not None else None,
                    **{str(k): v for k, v in t.params.items()},
                }
                for t in study.trials
            ]
        ).to_json(
            scenario_dir / "optuna_trials.json",
            orient="records",
            indent=2,
        )

    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=BASE_SEED + 10_000 * scenario_idx + 7),
        study_name=f"{scenario.key}_oracle_beta",
    )
    study.optimize(
        objective,
        n_trials=HPO_N_TRIALS,
        show_progress_bar=False,
        callbacks=[persist_hpo_state],
    )

    best_multiplier = float(study.best_trial.params["beta_multiplier"])
    best_proposal_size = int(study.best_trial.params["proposal_size"])
    best_beta = float(beta_init * best_multiplier)
    best_config = {
        "scenario": scenario.key,
        "reference_p0": float(p0_reference),
        "beta_init": float(beta_init),
        "beta0_formula": float(init_payload["beta0_formula"]),
        "beta0_laplace": float(init_payload["beta0_laplace"]),
        "sigma_t": float(sigma_t),
        "best_beta_multiplier": best_multiplier,
        "best_beta": best_beta,
        "best_proposal_size": int(best_proposal_size),
        "best_objective_value": float(study.best_value),
        "best_trial_number": int(study.best_trial.number),
        "n_trials": int(len(study.trials)),
    }

    final_eval = run_named_mcmc_checkpoint_study(
        scenario.problem,
        scenario.exact_p,
        config_specs=[
            {
                "label": "init",
                "config_id": "init",
                "beta": float(beta_init),
                "proposal_size": int(baseline_swap),
                "source": "pilot_init",
            },
            {
                "label": "oracle",
                "config_id": "oracle",
                "beta": float(best_beta),
                "proposal_size": int(best_proposal_size),
                "source": "oracle_hpo",
                "selected_by_objectives": ["optuna"],
            },
        ],
        sigma_t=sigma_t,
        estimation_points=ESTIMATION_POINTS,
        repeats=FINAL_EVAL_REPEATS,
        base_seed=BASE_SEED + 5_000_000 + 100_000 * scenario_idx,
        template_cfg=final_eval_cfg,
        n_jobs=FINAL_EVAL_N_JOBS,
    )

    oracle_results[scenario.key] = {
        "best_config": best_config,
        "trial_rows": list(trial_rows),
        "final_eval": final_eval,
        "init_payload": {
            key: value
            for key, value in init_payload.items()
            if key != "pilot_t"
        },
    }

    if SAVE_OUTPUTS and run_dir is not None:
        pd.DataFrame(trial_rows).to_json(
            scenario_dir / "hpo_trials.jsonl",
            orient="records",
            lines=True,
        )
        pd.DataFrame(final_eval["records"]).to_json(
            scenario_dir / "final_eval_records.jsonl",
            orient="records",
            lines=True,
        )
        pd.DataFrame(final_eval["summary"]).to_json(
            scenario_dir / "final_eval_summary.json",
            orient="records",
            indent=2,
        )
        (scenario_dir / "best_config.json").write_text(
            json.dumps(best_config, indent=2),
            encoding="utf-8",
        )
        (scenario_dir / "metadata.json").write_text(
            json.dumps(
                {
                    "scenario": scenario.key,
                    "scenario_display": scenario.description,
                    "exact_p": float(scenario.exact_p),
                    "known_significance_threshold": scenario.extra.get("known_significance_threshold"),
                    "application_setting_key": scenario.extra.get("application_setting_key"),
                    "reference_p0": float(p0_reference),
                    "init_payload": oracle_results[scenario.key]["init_payload"],
                    "hpo_settings": {
                        "n_trials": int(HPO_N_TRIALS),
                        "trial_repeats": int(HPO_TRIAL_REPEATS),
                        "objective_metric": str(HPO_OBJECTIVE_METRIC),
                        "beta_multiplier_low": float(HPO_BETA_MULTIPLIER_LOW),
                        "beta_multiplier_high": float(HPO_BETA_MULTIPLIER_HIGH),
                        "swap_choices": [int(v) for v in swap_choices],
                        "checkpoint": int(HPO_CHECKPOINT),
                    },
                    "final_eval_settings": final_eval["settings"],
                },
                indent=2,
            ),
            encoding="utf-8",
        )

    best_row = pd.DataFrame([best_config])
    final_summary_df = pd.DataFrame(final_eval["summary"]).sort_values(["checkpoint", "label"])
    display(best_row)
    display(final_summary_df[[
        "checkpoint",
        "label",
        "mean_estimate",
        "rmse",
        "mean_abs_log10_error",
        "mean_variance_estimate",
        "mean_q_tilt_tail_share",
        "mean_ess",
        "mean_acceptance_rate",
        "mean_weight_cv",
    ]])

[I 2026-04-11 19:54:52,921] A new study created in memory with name: gwas_additive_score_ultra_n60_oracle_beta


{
  "scenario": "gwas_additive_score_ultra_n60",
  "exact_p": 5.0966827770782916e-09,
  "application_setting_key": "gwas_threshold_suite",
  "known_significance_threshold": 5e-08,
  "reference_p0": 5e-08,
  "beta0_formula": 4.369745524021772,
  "beta0_laplace": 3.076171875,
  "q_target": 0.014953487812212205,
  "sigma_t": 2.092157688533463,
  "swap_choices": [
    1,
    2
  ],
  "baseline_swap": 2
}


[I 2026-04-11 19:57:19,105] Trial 0 finished with value: 5.0966827770782916e-09 and parameters: {'beta_multiplier': 0.10258938876468872, 'proposal_size': 1}. Best is trial 0 with value: 5.0966827770782916e-09.
[I 2026-04-11 19:59:58,024] Trial 1 finished with value: 5.0966827770782916e-09 and parameters: {'beta_multiplier': 0.025821041690589155, 'proposal_size': 2}. Best is trial 0 with value: 5.0966827770782916e-09.
[I 2026-04-11 20:01:33,833] Trial 2 finished with value: 5.0966827770782916e-09 and parameters: {'beta_multiplier': 0.3043628927612995, 'proposal_size': 1}. Best is trial 0 with value: 5.0966827770782916e-09.
[I 2026-04-11 20:04:08,917] Trial 3 finished with value: 3.78124146393369e-09 and parameters: {'beta_multiplier': 0.4623420758675806, 'proposal_size': 2}. Best is trial 3 with value: 3.78124146393369e-09.
[I 2026-04-11 20:06:44,071] Trial 4 finished with value: 5.0966827770782916e-09 and parameters: {'beta_multiplier': 0.014068724028216151, 'proposal_size': 2}. Best i

,scenario,reference_p0,beta_init,beta0_formula,beta0_laplace,sigma_t,best_beta_multiplier,best_beta,best_proposal_size,best_objective_value,best_trial_number,n_trials
0,gwas_additive_score_ultra_n60,5.000000e-08,3.076172,4.369746,3.076172,2.092158,0.830727,2.55546,1,1.614051e-10,6,10


,checkpoint,label,mean_estimate,rmse,mean_abs_log10_error,mean_variance_estimate,mean_q_tilt_tail_share,mean_ess,mean_acceptance_rate,mean_weight_cv
0,500000,init,5.215549e-09,1.272398e-09,0.094932,1.368374e-18,0.001231,555.114752,0.480134,30.463882
1,500000,oracle,5.805107e-09,1.887987e-09,0.112787,3.588369e-18,0.000312,2494.247517,0.686499,15.169760
2,1000000,init,4.912544e-09,9.590792e-10,0.063689,6.878192e-19,0.001236,686.627990,0.480737,37.789640
3,1000000,oracle,5.675549e-09,7.963262e-10,0.056871,1.666356e-18,0.000320,2658.868123,0.686480,21.213403
4,2500000,init,4.959186e-09,4.973690e-10,0.040087,3.338368e-19,0.001265,833.301650,0.480957,52.417637
5,2500000,oracle,5.062211e-09,5.777393e-10,0.043711,4.824591e-19,0.000276,6931.059973,0.686332,18.680836
6,5000000,init,5.074946e-09,2.262674e-10,0.015025,2.048893e-19,0.001311,1770.094674,0.481468,55.332312
7,5000000,oracle,4.820358e-09,3.777826e-10,0.028455,2.277239e-19,0.000272,10545.863286,0.686402,23.617625


...

,scenario,reference_p0,beta_init,beta0_formula,beta0_laplace,sigma_t,best_beta_multiplier,best_beta,best_proposal_size,best_objective_value,best_trial_number,n_trials
0,poisson_diffmeans_hep_sig_n200,3.000000e-07,3.560547,3.897236,3.560547,0.209964,0.797755,2.840443,8,1.223084e-08,3,10


,checkpoint,label,mean_estimate,rmse,mean_abs_log10_error,mean_variance_estimate,mean_q_tilt_tail_share,mean_ess,mean_acceptance_rate,mean_weight_cv
0,500000,init,3.160533e-07,1.025761e-07,0.140794,2.872337e-15,0.022607,417.225955,0.418600,38.967042
1,500000,oracle,2.517397e-07,4.200214e-08,0.064323,1.094513e-15,0.005942,861.057768,0.451198,29.262520
2,1000000,init,2.792748e-07,8.062990e-08,0.091543,2.826637e-15,0.022609,347.459585,0.418596,69.603571
3,1000000,oracle,2.537605e-07,2.944677e-08,0.036074,7.417826e-16,0.005705,1019.867325,0.450973,44.163972
4,2500000,init,2.621000e-07,2.250892e-08,0.034300,1.569551e-15,0.022525,320.388442,0.418567,98.053667
5,2500000,oracle,2.525936e-07,1.496113e-08,0.023864,2.618917e-16,0.005532,2523.301555,0.451177,38.465003
6,5000000,init,2.894062e-07,4.766382e-08,0.055344,7.291666e-16,0.022614,1034.335715,0.418465,81.511068
7,5000000,oracle,2.601044e-07,1.598591e-08,0.025496,1.189997e-16,0.005667,5559.844719,0.451175,28.395985


[I 2026-04-11 23:00:22,215] A new study created in memory with name: gwas_additive_score_above_n60_oracle_beta


{
  "scenario": "gwas_additive_score_above_n60",
  "exact_p": 6.283615119311404e-07,
  "application_setting_key": "gwas_threshold_suite",
  "known_significance_threshold": 5e-08,
  "reference_p0": 5e-08,
  "beta0_formula": 3.7789085964287663,
  "beta0_laplace": 4.353515625,
  "q_target": 0.014953487812212205,
  "sigma_t": 2.6508936680602577,
  "swap_choices": [
    1,
    2
  ],
  "baseline_swap": 2
}


[I 2026-04-11 23:02:59,126] Trial 0 finished with value: 3.344835202935915e-08 and parameters: {'beta_multiplier': 0.447900909226713, 'proposal_size': 2}. Best is trial 0 with value: 3.344835202935915e-08.
[I 2026-04-11 23:04:35,326] Trial 1 finished with value: 1.589355050190623e-07 and parameters: {'beta_multiplier': 0.1250154025632522, 'proposal_size': 1}. Best is trial 0 with value: 3.344835202935915e-08.
[I 2026-04-11 23:06:11,400] Trial 2 finished with value: 1.0161635405718658e-06 and parameters: {'beta_multiplier': 0.0048654455509106525, 'proposal_size': 1}. Best is trial 0 with value: 3.344835202935915e-08.
[I 2026-04-11 23:08:48,787] Trial 3 finished with value: 1.7080119123742765e-06 and parameters: {'beta_multiplier': 1.3937307519631434, 'proposal_size': 2}. Best is trial 0 with value: 3.344835202935915e-08.
[I 2026-04-11 23:10:24,926] Trial 4 finished with value: 4.897769068846827e-07 and parameters: {'beta_multiplier': 0.00297769633390416, 'proposal_size': 1}. Best is tri

## Review Saved Outputs

In [ ]:
if SAVE_OUTPUTS and run_dir is not None:
    print(f"Saved outputs under: {run_dir}")
    display(pd.DataFrame([
        {
            "scenario": key,
            "best_beta": payload["best_config"]["best_beta"],
            "best_multiplier": payload["best_config"]["best_beta_multiplier"],
            "best_proposal_size": payload["best_config"]["best_proposal_size"],
            "best_objective_value": payload["best_config"]["best_objective_value"],
        }
        for key, payload in oracle_results.items()
    ]))
else:
    print("SAVE_OUTPUTS=False, so no saved outputs to display.")

## Reload Saved Results Without Rerunning

In [ ]:
# RELOAD_BETA_DIR = None
# # Example:
# # RELOAD_BETA_DIR = project_root / "results" / "mcmcis_beta_notebook" / "20260411_120000_beta_diag" / "gwas_additive_score_sig_n60"

# if RELOAD_BETA_DIR is not None:
#     print(json.loads((RELOAD_BETA_DIR / "best_config.json").read_text()))
#     display(pd.read_json(RELOAD_BETA_DIR / "final_eval_summary.json"))
#     display(pd.read_json(RELOAD_BETA_DIR / "hpo_trials.jsonl", lines=True).sort_values("objective_value").head())
# else:
#     print("Set RELOAD_BETA_DIR to a saved oracle-beta directory to inspect saved results from disk only.")